Pregunta 11. Agregación con filtro – Costo del marketing inútil
Enunciado: Sabiendo que a la agencia (canal TA/TO) se le paga USD 1.5 por cada reserva. ¿Cuánto dinero en total se ha pagado a la agencia por reservas que finalmente fueron canceladas (is_canceled = 1)? Escribe la consulta que calcule este monto de pérdida.

In [0]:
%sql
SELECT
    COUNT(*) AS reservas_canceladas,
    COUNT(*) * 1.5 AS perdida_total_usd
FROM cursodataacademy.proyecto_hotel.hotel_bookings
WHERE distribution_channel = 'TA/TO'
AND is_canceled = 1;

conclusion: Se calculó el gasto destinado a las agencias por las reservas canceladas del canal TA/TO. Este monto representa una pérdida económica que debe considerarse al renegociar los contratos de marketing.

Pregunta 12. DATE_DIFF – Costo de cancelar tarde
Enunciado: Cuando un huésped cancela con menos de 3 días de anticipación respecto a su llegada, el hotel pierde USD 120. Escribe una consulta que cuente cuántas reservas cumplen esta condición (is_canceled = 1 y diferencia entre llegada y cancelación menor a 3 días) y multiplica esa cantidad por 120 para ver la pérdida total.

In [0]:
%sql
WITH reservas AS (
    SELECT
        is_canceled,
        TO_DATE(
            CONCAT(
                arrival_date_year, '-',
                arrival_date_month, '-',
                arrival_date_day_of_month
            ),
            'yyyy-MMMM-d'
        ) AS arrival_date,

        COALESCE(
            TO_DATE(TRY_TO_TIMESTAMP(reservation_status_date, 'M/d/yy')),
            TO_DATE(TRY_TO_TIMESTAMP(reservation_status_date, 'yyyy-MM-dd'))
        ) AS fecha_cancelacion

    FROM cursodataacademy.proyecto_hotel.hotel_bookings
)
SELECT
    COUNT(*) AS cancelaciones_tardias,
    COUNT(*) * 120 AS perdida_total_usd
FROM reservas
WHERE is_canceled = 1
AND fecha_cancelacion IS NOT NULL
AND DATEDIFF(arrival_date, fecha_cancelacion) BETWEEN 0 AND 2;

conclusion: Se identificaron las reservas canceladas con menos de tres días de anticipación y se calculó el costo operativo de USD 120 por cada caso. Este resultado permite cuantificar las pérdidas ocasionadas por cancelaciones tardías.

Pregunta 13. LEFT JOIN – Relación de reservas con países
Enunciado: Imagina que tenemos una segunda tabla llamada tb_paises con las columnas cod_pais y nombre_pais. Une tu tabla de reservas (usando su columna country) con tb_paises mediante un LEFT JOIN. Selecciona si la reserva fue cancelada y el nombre_pais.

SELECT
    r.is_canceled,
    r.country,
    p.nombre_pais
FROM cursodataacademy.proyecto_hotel.hotel_bookings r
LEFT JOIN cursodataacademy.proyecto_hotel.tb_paises p
    ON r.country = p.cod_pais;

conclusion: La consulta permite relacionar las reservas con los países de procedencia, identificando su estado de cancelación y conservando todos los registros de reservas, incluso cuando no existe coincidencia en la tabla de países.

Pregunta 14. INNER JOIN – Países con mayor cantidad de cancelaciones
Enunciado: Usando el mismo JOIN del ejercicio anterior, averigua cuáles son los cinco países que tienen la mayor cantidad de reservas canceladas.

SELECT
    p.nombre_pais,
    COUNT(*) AS reservas_canceladas
FROM cursodataacademy.proyecto_hotel.hotel_bookings r
INNER JOIN cursodataacademy.proyecto_hotel.tb_paises p
    ON r.country = p.cod_pais
WHERE r.is_canceled = 1
GROUP BY p.nombre_pais
ORDER BY reservas_canceladas DESC
LIMIT 5;

conclusion: La consulta permite determinar los cinco países con mayor cantidad de reservas canceladas, facilitando la identificación de los mercados que requieren mayor atención en las estrategias de gestión hotelera.

Pregunta 15. FULL OUTER JOIN – Relación de vendedores y reservas
Enunciado: Supongamos que Recursos Humanos tiene una tabla tb_vendedores (con id_vendedor y nombre_vendedor) y cruzamos las reservas para saber qué vendedor atendió qué reserva. Realiza un FULL OUTER JOIN entre las reservas y los vendedores. Selecciona el canal de distribución y el nombre del vendedor.

SELECT
    r.distribution_channel,
    v.nombre_vendedor
FROM cursodataacademy.proyecto_hotel.hotel_bookings r
FULL OUTER JOIN cursodataacademy.proyecto_hotel.tb_vendedores v
    ON r.id_vendedor = v.id_vendedor;

conclusion: La consulta permite relacionar las reservas con los vendedores, identificando posibles reservas sin vendedor asignado y vendedores que no tienen reservas asociadas, facilitando el control de la gestión comercial.

Pregunta 16. RIGHT JOIN – Relación de reservas y facturación
Enunciado: El departamento de contabilidad tiene una tabla de facturación (tb_facturas con id_reserva y monto_facturado). Usando un RIGHT JOIN desde las reservas hacia las facturas, garantiza que veamos todas las facturas emitidas, cruzadas con el tipo de hotel.

SELECT
    f.id_reserva,
    f.monto_facturado,
    r.hotel
FROM cursodataacademy.proyecto_hotel.hotel_bookings r
RIGHT JOIN cursodataacademy.proyecto_hotel.tb_facturas f
    ON r.id_reserva = f.id_reserva;

conclusion: La consulta permite relacionar las facturas emitidas con el tipo de hotel, conservando todos los registros de facturación, incluso cuando alguna factura no encuentra una reserva asociada.